In [1]:
import os 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

dir_path = '/space/gzanardini/Features_Ryan'

# get file list of all files in directory and subdirectories
file_list = []
for root, dirs, files in os.walk(dir_path):
    for file in files:
        if file.endswith('.csv'):
            file_list.append(os.path.join(root, file))  

print(len(file_list))

lengths = [ 2 ,  5 , 10 , 20 , 60 ,120 ,300]

df=pd.read_csv(file_list[0])
print(df.head())

150
    subject  epoch_length_sec  0_chmean_power_kurtosis_scale0  \
0  sub-0000                 2                        6.683000   
1  sub-0000                 5                        1.644788   
2  sub-0000                10                       -0.375567   
3  sub-0000                20                       -0.911975   
4  sub-0000                60                        0.215013   

   1_chmean_power_kurtosis_scale0  2_chmean_power_kurtosis_scale0  \
0                        9.546438                        3.169281   
1                        4.428618                        0.892126   
2                        0.830983                       -0.274939   
3                       -0.748091                       -1.154413   
4                       -0.725354                       -0.124355   

   3_chmean_power_kurtosis_scale0  4_chmean_power_kurtosis_scale0  \
0                       18.730960                       58.915109   
1                        0.919699                   

In [2]:
#dictionary with mapings for feature names and combiners 
feature_map = {
    'kurtosis': 'kurt',
    'skewness': 'skew',
    'connectivity': 'cc'
}
    

In [3]:
#utm_CAR_5s_skew
outpath = '/space/gzanardini/emc_hv/'
## for each file in file list, parse the file name to get montage, feature name and combiner
data = []
for file in file_list:
    file_name = os.path.basename(file)
    parts = file_name.split('/')[-1].split('_')
    combiner = parts[-1].replace('.csv', '')

    if 'power' not in parts:
        feature_name = parts[-4]
        montage = parts[-5]
    else:
        feature_name = 'sr' if 'relative' in parts else 'sa'
        montage = parts[2]

    if combiner in feature_map.keys():
        combiner = feature_map[combiner]
    if feature_name in feature_map.keys():
        feature_name = feature_map[feature_name]   
        
    if feature_name == 'stockwell':
        continue 
    for length in lengths: #epoch_length_sec
        print(f'Feature: {feature_name}, Montage: {montage}, Combiner: {combiner}, Length: {length}')       
        df_2 = df[df['epoch_length_sec'] == length]
        if len(df_2)<139:
            print(f'Not enough data points for length {length}, only {len(df_2)}')
            continue
        np.save(os.path.join(outpath, f'{feature_name}_{montage}_{length}s_{combiner}.npy'), df_2.drop(columns=['subject', 'epoch_length_sec']).values)




Feature: cwt, Montage: CAR, Combiner: kurt, Length: 2
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 5
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 10
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 20
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 60
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 120
Feature: cwt, Montage: CAR, Combiner: kurt, Length: 300
Not enough data points for length 300, only 1
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 2
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 5
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 10
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 20
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 60
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 120
Feature: cwt, Montage: Laplacian, Combiner: std, Length: 300
Not enough data points for length 300, only 1
Feature: cwt, Montage: Cz, Combiner: std, Length: 2
Feature: cwt, Montage: Cz, Combiner: std, Length: 5

In [4]:
outpath = '/space/gzanardini/emc_hv/'
## for each file in file list, parse the file name to get montage, feature name and combiner
data = []
for file in file_list:

    file_name = os.path.basename(file)
    parts = file_name.split('/')[-1].split('_')
    combiner = parts[-1].replace('.csv', '')

  
    feature_name = parts[-4]
    montage = parts[-5]

    if feature_name != 'stockwell':
        continue 

    print(feature_name, montage, combiner)

    raw_df=pd.read_csv(file)

    # separate the content of the columns into two dataframes, keep the first two columns as they are
    sst = raw_df.iloc[:, :2]
    mst = raw_df.iloc[:, :2]

    # in the 1st append the columns which contain _sr_ in their name
    sst = pd.concat([sst, raw_df.filter(like='_sr_')], axis=1)

    # in the 2nd append the columns which contain _power_skew_ in their name
    mst = pd.concat([mst, raw_df.filter(like='_power_skew_')], axis=1)

    #print columns of both dataframes
    # print(sst.columns)
    # print(mst.columns)

    for length in lengths: #epoch_length_sec
        # for sst_epochs only keep the columns that end in _sr_mean_{combiner} and for mst_epochs only keep the columns that end in _power_skew_{combiner}
        sst_epoch = sst[sst['epoch_length_sec'] == length]
        mst_epoch = mst[mst['epoch_length_sec'] == length]

        sst_filtered = sst_epoch.filter(regex=f'_sr_mean_{combiner}$')
        mst_filtered = mst_epoch.filter(regex=f'_power_skew_{combiner}$')
        
        # add back the subject and epoch_length_sec columns
        sst_filtered = pd.concat([sst_epoch[['subject', 'epoch_length_sec']], sst_filtered], axis=1)
        mst_filtered = pd.concat([mst_epoch[['subject', 'epoch_length_sec']], mst_filtered], axis=1)

        print(f'Len SST: {len(sst_filtered.columns)}, Len MST: {len(mst_filtered.columns)}')
        print(len(sst_filtered), len(mst_filtered))
    #/print(sst_filtered.head(0), mst_filtered.head(0))

        if len(sst_epoch)<139 or len(mst_epoch)<139:
            print(f'Not enough data points for length {length}, only {len(sst_epoch)} and {len(mst_epoch)}')
            continue


        np.save(os.path.join(outpath, f'sst_{montage}_{length}s_{combiner}.npy'), sst_epoch.drop(columns=['subject', 'epoch_length_sec']).values)
        np.save(os.path.join(outpath, f'mst_{montage}_{length}s_{combiner}.npy'), mst_epoch.drop(columns=['subject', 'epoch_length_sec']).values)

    #   np.save(os.path.join(outpath, f'{feature_name}_{montage}_{length}s_{combiner}.npy'), mst_epoch.drop(columns=['subject', 'epoch_length_sec']).values)


stockwell Laplacian median
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
139 139
Len SST: 97, Len MST: 97
1 1
Not enough data points for length 300, only 1 and 1
stockwell Cz kurtosis
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
1 1
Not enough data points for length 300, only 1 and 1
stockwell Cz mean
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
139 139
Len SST: 112, Len MST: 112
1 1
Not enough data points for length 300, only 1 and 1
stockwell Laplacian std
Len SST: 97, Len MST: 97
139 139
Len SST: 97

In [48]:
# check that for all files in the file list the first two columns are 'subject' and 'epoch_length_sec', also check that they all have the same length

df0=pd.read_csv(file_list[0])
# remove the lines with epoch_length_sec=300
df0 = df0[df0['epoch_length_sec'] != 300]

for i,file in enumerate(file_list):
    file_name = os.path.basename(file)
    df=pd.read_csv(file)
    df=df[df['epoch_length_sec'] != 300]

    parts = file_name.split('/')[-1].split('_')

    if not all(df.columns[:2] == ['subject', 'epoch_length_sec']):
        print(f"File {file_name} does not have the correct columns.")

    if parts[3]=='plv':
        continue
    
    if len(df) != len(df0):
        print(f'File {i} ,{"_".join(parts)}')
        print(f'File contains {len(df)} rows, while the reference file contains {len(df0)} rows.')

    # get unique values of the subject column
    subjects=df['subject'].unique()


In [49]:
print(subjects)

['sub-0000' 'sub-0001' 'sub-0002' 'sub-0004' 'sub-0005' 'sub-0006'
 'sub-0007' 'sub-0008' 'sub-0009' 'sub-0010' 'sub-0012' 'sub-0013'
 'sub-0014' 'sub-0015' 'sub-0016' 'sub-0017' 'sub-0018' 'sub-0019'
 'sub-0020' 'sub-0021' 'sub-0023' 'sub-0024' 'sub-0025' 'sub-0026'
 'sub-0027' 'sub-0028' 'sub-0029' 'sub-0030' 'sub-0032' 'sub-0033'
 'sub-0034' 'sub-0035' 'sub-0036' 'sub-0037' 'sub-0038' 'sub-0039'
 'sub-0040' 'sub-0041' 'sub-0042' 'sub-0043' 'sub-0044' 'sub-0045'
 'sub-0046' 'sub-0047' 'sub-0048' 'sub-0049' 'sub-0050' 'sub-0051'
 'sub-0052' 'sub-0053' 'sub-0054' 'sub-0055' 'sub-0056' 'sub-0057'
 'sub-0058' 'sub-0059' 'sub-0060' 'sub-0061' 'sub-0062' 'sub-0064'
 'sub-0065' 'sub-0066' 'sub-0067' 'sub-0068' 'sub-0069' 'sub-0070'
 'sub-0071' 'sub-0072' 'sub-0073' 'sub-0074' 'sub-0075' 'sub-0076'
 'sub-0077' 'sub-0078' 'sub-0079' 'sub-0080' 'sub-0081' 'sub-0082'
 'sub-0083' 'sub-0084' 'sub-0085' 'sub-0086' 'sub-0087' 'sub-0088'
 'sub-0089' 'sub-0090' 'sub-0091' 'sub-0092' 'sub-0093' 'sub-0

In [ ]:
#open emc_lookup.csv from /space/gzanardini/
emc_lookup=pd.read_csv('/space/gzanardini/emc_lookup.csv')
# rename colulmn Research_Code to subject, and Outcome_Label to epilepsy
emc_lookup = emc_lookup.rename(columns={'Research_Code': 'subject', 'Outcome_Label': 'epilepsy'})
# now rename the subjects  from ML_EEG_TUD_#### to sub-#### 
emc_lookup['subject'] = emc_lookup['subject'].str.replace('ML_EEG_TUD_', 'sub-')

In [53]:
# add to subjects a columns with the values in emc_lookup['epilepsy'] for the the rows with matching subjects, use int dtype
subjects = pd.DataFrame(subjects, columns=['subject'])
subjects = subjects.merge(emc_lookup[['subject', 'epilepsy']], on='subject', how='left')
subjects['epilepsy'] = subjects['epilepsy'].astype(int)
print(subjects)

# save in the same folder as the features, as a csv file
subjects.to_csv(os.path.join(outpath, 'description.csv'), index=False)


      subject  epilepsy
0    sub-0000         0
1    sub-0001         0
2    sub-0002         0
3    sub-0004         1
4    sub-0005         0
..        ...       ...
134  sub-0141         1
135  sub-0142         0
136  sub-0143         0
137  sub-0144         0
138  sub-0145         0

[139 rows x 2 columns]


     Research_Code  Outcome_Label
0  ML_EEG_TUD_0000              0
1  ML_EEG_TUD_0001              0
2  ML_EEG_TUD_0002              0
3  ML_EEG_TUD_0003              0
4  ML_EEG_TUD_0004              1
           subject  epilepsy
0  ML_EEG_TUD_0000         0
1  ML_EEG_TUD_0001         0
2  ML_EEG_TUD_0002         0
3  ML_EEG_TUD_0003         0
4  ML_EEG_TUD_0004         1
    subject  epilepsy
0  sub-0000         0
1  sub-0001         0
2  sub-0002         0
3  sub-0003         0
4  sub-0004         1
